# EarthScape: ML exploration
Explores the real FAO temperature data and reproduces the temperature-trend model evaluation used in the app (`climatechanges/ml.py`).

Run from the project folder: `jupyter notebook notebooks/ml_exploration.ipynb`

In [ ]:
import pandas as pd, numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('../data/Temperature.csv', keep_default_na=False, na_values=[''])
years = [c for c in df.columns if c.isdigit()]
world = df[df.Country == 'World'][years].T.reset_index()
world.columns = ['year', 'anomaly']
world['year'] = world.year.astype(int)
world = world.dropna()
world.tail()

## Warming per decade by region (real data)

In [ ]:
regions = ['World', 'Africa', 'Americas', 'Asia', 'Europe', 'Oceania']
long = df[df.Country.isin(regions)].melt(id_vars='Country', value_vars=years, var_name='year', value_name='anomaly').dropna()
long['year'] = long.year.astype(int)
{c: round(np.polyfit(g.year, g.anomaly.astype(float), 1)[0] * 10, 3) for c, g in long.groupby('Country')}

## Time-based hold-out: train on all but the last 10 years

In [ ]:
X, y = world[['year']].values.astype(float), world.anomaly.values.astype(float)
Xtr, Xte, ytr, yte = X[:-10], X[-10:], y[:-10], y[-10:]
models = {
    'Linear': LinearRegression(),
    'Poly2 + Ridge': make_pipeline(StandardScaler(), PolynomialFeatures(2), Ridge(alpha=1.0)),
    'Random forest': RandomForestRegressor(n_estimators=200, random_state=42),
}
rows = []
for name, m in models.items():
    p = m.fit(Xtr, ytr).predict(Xte)
    rows.append({'model': name, 'MAE': mean_absolute_error(yte, p), 'RMSE': mean_squared_error(yte, p) ** 0.5, 'R2': r2_score(yte, p)})
pd.DataFrame(rows).round(3)

## Correlation: global temperature vs climate disasters (real data)

In [ ]:
from scipy import stats
d = pd.read_csv('../data/Disaster.csv', encoding='utf-8-sig', keep_default_na=False, na_values=[''])
d = d[d.Indicator.str.endswith('Number of Disasters: TOTAL') & (d.ISO2 != '')]
dy = [c for c in d.columns if c.isdigit()]
per_year = d[dy].apply(pd.to_numeric, errors='coerce').sum().rename('disasters')
per_year.index = per_year.index.astype(int)
m = world.set_index('year').join(per_year, how='inner')
stats.pearsonr(m.anomaly, m.disasters)